# 🚀 YOLOv8 Intrusion Detection Training (Kaggle)

**BEFORE RUNNING:**
1. Go to **Settings → Accelerator → GPU T4 x2**
2. Add your `intrusion-detection` dataset via **+ Add Input**

**Estimated Time: ~2-3 hours** ✅

In [ ]:
# ============== CELL 1: Install & Check GPU ==============
!pip install -q ultralytics
!nvidia-smi
print("✅ GPU ready!")

In [ ]:
# ============== CELL 2: Setup Dataset ==============
import os
import shutil
import json

# Find dataset
KAGGLE_INPUT = "/kaggle/input"
DATASET_NAME = None
for name in os.listdir(KAGGLE_INPUT):
    if 'intrusion' in name.lower():
        DATASET_NAME = name
        break

if not DATASET_NAME:
    raise Exception("❌ Dataset not found! Add intrusion-detection dataset via '+ Add Input'")

KAGGLE_DATASET = f"{KAGGLE_INPUT}/{DATASET_NAME}"
LOCAL_DATASET = "/kaggle/working/dataset"

# Copy dataset
if not os.path.exists(LOCAL_DATASET):
    print("📦 Copying dataset...")
    shutil.copytree(KAGGLE_DATASET, LOCAL_DATASET)
print(f"✅ Dataset ready at: {LOCAL_DATASET}")

# Check structure
for split in ['train', 'valid', 'test']:
    path = f"{LOCAL_DATASET}/{split}"
    if os.path.exists(path):
        imgs = len([f for f in os.listdir(path) if f.endswith('.jpg')])
        print(f"   {split}: {imgs} images")

In [ ]:
# ============== CELL 3: Convert COCO to YOLO Format ==============
import json
import os
import shutil
from pathlib import Path

def coco_to_yolo(dataset_path):
    """Convert COCO annotations to YOLO format with proper folder structure"""
    
    for split in ['train', 'valid', 'test']:
        split_path = Path(dataset_path) / split
        json_file = split_path / '_annotations.coco.json'
        
        if not json_file.exists():
            print(f"⚠️ No annotations for {split}")
            continue
        
        # Create proper YOLOv8 structure: images/ and labels/
        images_dir = split_path / 'images'
        labels_dir = split_path / 'labels'
        images_dir.mkdir(exist_ok=True)
        labels_dir.mkdir(exist_ok=True)
        
        # Load COCO annotations
        with open(json_file, 'r') as f:
            coco = json.load(f)
        
        # Build image id to filename mapping
        img_map = {img['id']: img for img in coco['images']}
        
        # Build category id mapping (make 0-indexed)
        cat_ids = sorted([c['id'] for c in coco['categories']])
        cat_map = {old_id: new_id for new_id, old_id in enumerate(cat_ids)}
        
        # Group annotations by image
        img_anns = {}
        for ann in coco['annotations']:
            img_id = ann['image_id']
            if img_id not in img_anns:
                img_anns[img_id] = []
            img_anns[img_id].append(ann)
        
        # Move images and create labels
        moved_count = 0
        for img_id, img_info in img_map.items():
            img_name = img_info['file_name']
            img_w, img_h = img_info['width'], img_info['height']
            
            # Move image to images/ subfolder
            src_img = split_path / img_name
            dst_img = images_dir / img_name
            if src_img.exists() and not dst_img.exists():
                shutil.move(str(src_img), str(dst_img))
                moved_count += 1
            
            # Create label file
            label_name = Path(img_name).stem + '.txt'
            label_path = labels_dir / label_name
            
            # Get annotations for this image
            anns = img_anns.get(img_id, [])
            
            if anns:
                lines = []
                for ann in anns:
                    x, y, w, h = ann['bbox']  # COCO format: x,y,w,h (top-left)
                    
                    # Convert to YOLO format: x_center, y_center, width, height (normalized)
                    x_center = (x + w/2) / img_w
                    y_center = (y + h/2) / img_h
                    w_norm = w / img_w
                    h_norm = h / img_h
                    
                    # Clamp values to [0, 1]
                    x_center = max(0, min(1, x_center))
                    y_center = max(0, min(1, y_center))
                    w_norm = max(0, min(1, w_norm))
                    h_norm = max(0, min(1, h_norm))
                    
                    class_id = cat_map[ann['category_id']]
                    lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
                
                with open(label_path, 'w') as f:
                    f.write('\n'.join(lines))
        
        # Count results
        num_labels = len(list(labels_dir.glob('*.txt')))
        num_images = len(list(images_dir.glob('*.jpg'))) + len(list(images_dir.glob('*.png')))
        print(f"✅ {split}: {num_images} images, {num_labels} labels")
    
    # Get class names
    with open(Path(dataset_path) / 'train' / '_annotations.coco.json', 'r') as f:
        coco = json.load(f)
    class_names = [c['name'] for c in sorted(coco['categories'], key=lambda x: x['id'])]
    
    return class_names

# Convert
print("🔄 Converting COCO to YOLO format...")
class_names = coco_to_yolo(LOCAL_DATASET)
print(f"\n📋 Classes: {class_names}")

# Verify labels exist
print("\n🔍 Verification:")
import subprocess
subprocess.run(f"ls -la {LOCAL_DATASET}/train/images/ | head -5", shell=True)
subprocess.run(f"ls -la {LOCAL_DATASET}/train/labels/ | head -5", shell=True)

In [ ]:
# ============== CELL 4: Create data.yaml ==============
import yaml

# YOLOv8 expects paths to images folders
data_yaml = {
    'path': LOCAL_DATASET,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {i: name for i, name in enumerate(class_names)}
}

yaml_path = f"{LOCAL_DATASET}/data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✅ Created {yaml_path}")
print("\n📄 Contents:")
with open(yaml_path, 'r') as f:
    print(f.read())

In [ ]:
# ============== CELL 5: TRAIN YOLOv8 🚀 ==============
from ultralytics import YOLO

# Load pretrained model
model = YOLO('yolov8s.pt')  # Small model - good balance of speed/accuracy

# Train
print("="*60)
print("🚀 Starting YOLOv8 Training...")
print("="*60)

results = model.train(
    data=f"{LOCAL_DATASET}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=10,  # Early stopping
    save=True,
    project='/kaggle/working',
    name='intrusion_detection',
    exist_ok=True
)

print("\n" + "="*60)
print("✅ Training Complete!")
print("="*60)

In [ ]:
# ============== CELL 6: Evaluate & Export ==============
import os

# Find best model
best_model = '/kaggle/working/intrusion_detection/weights/best.pt'

if os.path.exists(best_model):
    print(f"✅ Best model: {best_model}")
    size = os.path.getsize(best_model) / (1024*1024)
    print(f"   Size: {size:.1f} MB")
    
    # Load and evaluate
    model = YOLO(best_model)
    
    print("\n📊 Validation Results:")
    metrics = model.val()
    print(f"   mAP50: {metrics.box.map50:.3f}")
    print(f"   mAP50-95: {metrics.box.map:.3f}")
    
    # Export to ONNX (for deployment)
    print("\n📦 Exporting to ONNX...")
    model.export(format='onnx', imgsz=640)
    print("✅ ONNX model exported!")
else:
    print("❌ Model not found. Check training output.")

# List output files
print("\n📁 Output files:")
!ls -la /kaggle/working/intrusion_detection/weights/

In [ ]:
# ============== CELL 7: Download Instructions ==============
print("="*60)
print("📥 TO DOWNLOAD YOUR MODEL:")
print("="*60)
print("\n1. Click 'Save Version' (top right)")
print("2. Select 'Save & Run All'")
print("3. Go to 'Output' tab after completion")
print("4. Download from: intrusion_detection/weights/")
print("   - best.pt (best model)")
print("   - best.onnx (for deployment)")
print("\n" + "="*60)